# IRWA Final Project. Part 3: Ranking and Filtering

Course: Information Retrieval and Web Analytics (IRWA)
Group members: Sergi Cases, Julia Garcia, Martí Casas

Top 20 results of the 5 test queries of Part 2 with four rankings over the conjunctive (AND) results: TF-IDF + cosine, BM25, our custom score and Word2Vec + cosine.

- src/indexing.py: corpus loading, inverted index, idf, document lengths, BM25 idf, product info and AND search
- src/ranking.py: rank_tfidf, rank_bm25, rank_custom
- src/word2vec_ranking.py: train_word2vec, compute_doc_vectors, rank_word2vec

## 0. Setup

In [1]:
from itertools import combinations

import pandas as pd
from IPython.display import display
from scipy.stats import kendalltau

from src.indexing import (load_corpus, build_index, compute_idf, compute_doc_norms, compute_doc_lengths,
                          compute_bm25_idf, get_product_info, conjunctive_search)
from src.ranking import rank_tfidf, rank_bm25, rank_custom
from src.word2vec_ranking import train_word2vec, compute_doc_vectors, rank_word2vec

DATA_DIR = "./"
CORPUS_PATH = DATA_DIR + "processed_corpus.json"
RANKINGS_PATH = DATA_DIR + "part3_rankings.csv"

TOP_K = 20
TEST_QUERIES = [
    "women cotton kurta",
    "men slim fit chinos",
    "hooded full sleeve sweatshirt",
    "blue denim shorts",
    "women black leggings",
]
METHODS = ["tfidf", "bm25", "custom", "word2vec"]

pd.set_option("display.max_colwidth", 70)

## 1. Build the index and the models

In [2]:
%%time
corpus = load_corpus(CORPUS_PATH)
index = build_index(corpus, field="all_terms")
idf = compute_idf(index, len(corpus))
doc_norms = compute_doc_norms(index, idf)

doc_lens = compute_doc_lengths(corpus)
avg_len = sum(doc_lens.values()) / len(doc_lens)
idf_bm25 = compute_bm25_idf(index, len(corpus))
product_info = get_product_info(corpus)

w2v_model = train_word2vec(corpus)
doc_vectors = compute_doc_vectors(corpus, w2v_model)

print("Documents:", len(corpus), "| Vocabulary:", len(index), "| Average length:", round(avg_len, 1),
      "| Word2Vec vocabulary:", len(w2v_model.wv))

Documents: 28080 | Vocabulary: 5335 | Average length: 39.5 | Word2Vec vocabulary: 4187
CPU times: total: 6.3 s
Wall time: 6.23 s


## 2. Rankings of the test queries

In [3]:
def rank_all(query):
    """Full ranking of the AND results of a query with every method: method -> [(pid, score)]."""
    pids = conjunctive_search(query, index)
    return {
        "tfidf": rank_tfidf(query, pids, index, idf, doc_norms),
        "bm25": rank_bm25(query, pids, index, idf_bm25, doc_lens, avg_len),
        "custom": rank_custom(query, pids, index, idf_bm25, doc_lens, avg_len, product_info),
        "word2vec": rank_word2vec(query, pids, w2v_model, doc_vectors),
    }


products = corpus.set_index("pid")[["title", "average_rating_num", "selling_price_num", "discount_num"]]
products.columns = ["title", "rating", "price", "discount"]

rankings = {q: rank_all(q) for q in TEST_QUERIES}

rows = []
for qid, query in enumerate(TEST_QUERIES, start=1):
    for method in METHODS:
        for rank, (pid, score) in enumerate(rankings[query][method][:TOP_K], start=1):
            rows.append({"query_id": qid, "query": query, "method": method, "rank": rank, "pid": pid,
                         **products.loc[pid].to_dict(), "score": score})
results = pd.DataFrame(rows)[["query_id", "query", "method", "rank", "pid", "title", "score",
                              "rating", "price", "discount"]]
results.to_csv(RANKINGS_PATH, index=False, encoding="utf-8-sig")
print("Saved", len(results), "rows to", RANKINGS_PATH)
pd.DataFrame({"query": TEST_QUERIES, "AND results": [len(rankings[q]["tfidf"]) for q in TEST_QUERIES]})

Saved 372 rows to ./part3_rankings.csv


,query,AND results
0,women cotton kurta,358
1,men slim fit chinos,186
2,hooded full sleeve sweatshirt,497
3,blue denim shorts,101
4,women black leggings,13


In [4]:
def show_query(qid):
    query = TEST_QUERIES[qid - 1]
    for method in METHODS:
        print(f"Query {qid}: '{query}' | {method}")
        table = results[(results["query_id"] == qid) & (results["method"] == method)]
        display(table[["rank", "pid", "title", "score", "rating", "price", "discount"]]
                .set_index("rank").round({"score": 4}))

### 2.1 Query 1: women cotton kurta

In [5]:
show_query(1)

Query 1: 'women cotton kurta' | tfidf


,pid,title,score,rating,price,discount
rank,,,,,,
1,ETHF25572QHXAGEQ,Women Kurta and Pyjama Set Cotton Blend,0.7175,4.8,823.0,36.0
2,ETHF2557ZHWRYHPT,Women Kurta and Pyjama Set Cotton Blend,0.7175,4.0,848.0,34.0
3,ETHF2557CGF4CUMW,Women Kurta and Pyjama Set Cotton Blend,0.7101,4.0,848.0,15.0
4,ETHF2557EGMJTGFT,Women Kurta and Pyjama Set Cotton Blend,0.7062,4.0,848.0,15.0
5,ETHF2557SGSWZEVH,Women Kurta and Pyjama Set Cotton Blend,0.7008,4.8,848.0,34.0
6,KTAFD2U8FHZRECCF,Women Self Design Cotton Blend Straight Kurta (Black),0.6646,3.7,599.0,76.0
7,KTAFD2U8ZGHHHNYF,Women Self Design Cotton Blend Straight Kurta (Black),0.6646,3.7,599.0,76.0
8,KTAF5SD3WHM2V9HZ,Women Printed Cotton Blend Straight Kurta (Orange),0.6607,4.1,599.0,33.0
9,KTAF5SD2C3TJACQF,Women Striped Cotton Blend Straight Kurta (Pink),0.6492,3.0,540.0,39.0


Query 1: 'women cotton kurta' | bm25


,pid,title,score,rating,price,discount
rank,,,,,,
1,KTAFZPA5GRMYWAJ9,Women Striped Cotton Blend Straight Kurta (Yellow),7.8175,4.3,450.0,70.0
2,KTAF85TUHDNZCU3P,Women Solid Cotton Blend Straight Kurta (Purple),7.7688,4.0,538.0,79.0
3,KTAFTQVJG6MZJYMS,Women Printed Pure Cotton Straight Kurta (Brown),7.7221,4.1,629.0,64.0
4,KTAF8FJCZVQ7QHH9,Women Solid Cotton Blend Pathani Kurta (Black),7.6612,4.0,660.0,74.0
5,KTAF8FKVGGDFBHAE,Women Solid Cotton Blend Pathani Kurta (Pink),7.6612,3.6,647.0,75.0
6,KTAF8FJVVYE2EDWN,Women Solid Cotton Blend Straight Kurta (Yellow),7.6348,3.8,719.0,72.0
7,ETHFT9HQRGEXHETK,Women Kurta and Pyjama Set Cotton Blend,7.6082,3.0,1035.0,35.0
8,KTAF5SD2C3TJACQF,Women Striped Cotton Blend Straight Kurta (Pink),7.5915,3.0,540.0,39.0
9,KTAF5SD3WHM2V9HZ,Women Printed Cotton Blend Straight Kurta (Orange),7.5915,4.1,599.0,33.0


Query 1: 'women cotton kurta' | custom


,pid,title,score,rating,price,discount
rank,,,,,,
1,KTAFZPA5GRMYWAJ9,Women Striped Cotton Blend Straight Kurta (Yellow),1.3175,4.3,450.0,70.0
2,KTAF85TUHDNZCU3P,Women Solid Cotton Blend Straight Kurta (Purple),1.2878,4.0,538.0,79.0
3,KTAFTQVJG6MZJYMS,Women Printed Pure Cotton Straight Kurta (Brown),1.2649,4.1,629.0,64.0
4,KTAFZFX7HTMGQCG6,Women Solid Cotton Blend Straight Kurta (Black),1.2534,5.0,390.0,64.0
5,KTAFZFX7MGG3ZTYP,Women Solid Cotton Blend Straight Kurta (Red),1.2534,5.0,390.0,64.0
6,KTAFWG3KHPEVYFRG,Women Solid Pure Cotton Straight Kurta (Maroon),1.2497,4.7,699.0,65.0
7,KTAFWG3KMZYPJ7NJ,Women Solid Pure Cotton Straight Kurta (Green),1.2497,4.7,699.0,65.0
8,KTAFWG3KPKEGYRHG,Women Solid Pure Cotton Straight Kurta (Orange),1.2497,4.7,699.0,65.0
9,KTAFWG3KZYVZVGQH,Women Solid Pure Cotton Straight Kurta (Pink),1.2497,4.7,699.0,65.0


Query 1: 'women cotton kurta' | word2vec


,pid,title,score,rating,price,discount
rank,,,,,,
1,ETHF25572QHXAGEQ,Women Kurta and Pyjama Set Cotton Blend,0.9102,4.8,823.0,36.0
2,ETHF2557ZHWRYHPT,Women Kurta and Pyjama Set Cotton Blend,0.9102,4.0,848.0,34.0
3,ETHF2557EGMJTGFT,Women Kurta and Pyjama Set Cotton Blend,0.9085,4.0,848.0,15.0
4,ETHF2557SGSWZEVH,Women Kurta and Pyjama Set Cotton Blend,0.9072,4.8,848.0,34.0
5,ETHF2557CGF4CUMW,Women Kurta and Pyjama Set Cotton Blend,0.8983,4.0,848.0,15.0
6,KTAFCVTAWYHNGXGB,Women Striped Cotton Blend Straight Kurta (Green),0.8964,3.3,719.0,72.0
7,KTAF5SD3WHM2V9HZ,Women Printed Cotton Blend Straight Kurta (Orange),0.8953,4.1,599.0,33.0
8,KTAF5SD2C3TJACQF,Women Striped Cotton Blend Straight Kurta (Pink),0.8944,3.0,540.0,39.0
9,ETHFY7FBXY6ZDTKN,Women Kurta and Pyjama Set Pure Cotton,0.8908,NaN,724.0,51.0


### 2.2 Query 2: men slim fit chinos

In [6]:
show_query(2)

Query 2: 'men slim fit chinos' | tfidf


,pid,title,score,rating,price,discount
rank,,,,,,
1,TROF9F59KRE4HQ2E,Slim Fit Men Grey Cotton Blend Trousers,0.5365,3.5,1249.0,50.0
2,TROE9B6GEHCE4YFP,Slim Fit Men Multicolor Cotton Blend Trousers,0.4844,3.5,999.0,44.0
3,TRODYPCYVYUZH3MZ,Slim Fit Men Yellow Cotton Blend Trousers,0.4834,3.7,699.0,70.0
4,TROFWYDEQFHWGCTD,Slim Fit Men Blue Cotton Blend Trousers,0.4777,3.6,799.0,60.0
5,TROE9B6HRHMFTREG,Slim Fit Men Multicolor Cotton Blend Trousers,0.4768,3.5,1499.0,44.0
6,TROE9B6JVKFH8WTM,Slim Fit Men Multicolor Cotton Blend Trousers,0.4768,3.5,999.0,44.0
7,TROE9B6MPTHDJ8ZS,Slim Fit Men Multicolor Cotton Blend Trousers,0.4768,3.2,1499.0,44.0
8,TROE9B6MVJFCM9NJ,Slim Fit Men Multicolor Cotton Blend Trousers,0.4768,2.8,1499.0,44.0
9,TROE9B6Z8HUSPXXY,Slim Fit Men Multicolor Cotton Blend Trousers,0.4768,2.8,1499.0,44.0


Query 2: 'men slim fit chinos' | bm25


,pid,title,score,rating,price,discount
rank,,,,,,
1,TRODYPCYVYUZH3MZ,Slim Fit Men Yellow Cotton Blend Trousers,10.0218,3.7,699.0,70.0
2,TRODZZMJYWQTEGRY,Rocky Slim Fit Men Black Cotton Blend Trousers,9.9029,3.6,499.0,44.0
3,TROE3N2FAAGV3HSG,Rocky Slim Fit Men Beige Cotton Blend Trousers,9.9029,3.6,450.0,49.0
4,TROE3N2FYUQWF8DX,Rocky Slim Fit Men Beige Cotton Blend Trousers,9.9029,3.6,496.0,44.0
5,TROE3N2GXGYWPZ3H,Rocky Slim Fit Men Silver Cotton Blend Trousers,9.9029,3.6,456.0,49.0
6,TROE9B6GEHCE4YFP,Slim Fit Men Multicolor Cotton Blend Trousers,9.9029,3.5,999.0,44.0
7,TROEFUFU3DCZFBHG,"Slim Fit Men Maroon, Black Cotton Blend Trousers",9.8334,3.4,999.0,44.0
8,TROE3N2GDGHXPYYJ,"Rocky Slim Fit Men Silver, Beige Cotton Blend Trousers",9.7648,3.3,999.0,44.0
9,TROE9B6HRHMFTREG,Slim Fit Men Multicolor Cotton Blend Trousers,9.6971,3.5,1499.0,44.0


Query 2: 'men slim fit chinos' | custom


,pid,title,score,rating,price,discount
rank,,,,,,
1,TRODYPCYVYUZH3MZ,Slim Fit Men Yellow Cotton Blend Trousers,1.2725,3.7,699.0,70.0
2,TROE3N2FAAGV3HSG,Rocky Slim Fit Men Beige Cotton Blend Trousers,1.2164,3.6,450.0,49.0
3,TROE3N2GXGYWPZ3H,Rocky Slim Fit Men Silver Cotton Blend Trousers,1.2164,3.6,456.0,49.0
4,TRODZZMJYWQTEGRY,Rocky Slim Fit Men Black Cotton Blend Trousers,1.2115,3.6,499.0,44.0
5,TROE3N2FYUQWF8DX,Rocky Slim Fit Men Beige Cotton Blend Trousers,1.2115,3.6,496.0,44.0
6,TROE9B6GEHCE4YFP,Slim Fit Men Multicolor Cotton Blend Trousers,1.2042,3.5,999.0,44.0
7,TROEFUFU3DCZFBHG,"Slim Fit Men Maroon, Black Cotton Blend Trousers",1.1809,3.4,999.0,44.0
8,TROEC8RPUZYGFDZF,Slim Fit Men Multicolor Cotton Blend Trousers,1.1638,3.6,1499.0,44.0
9,TROEC9FGJDNTYREE,Slim Fit Men Multicolor Cotton Blend Trousers,1.1601,3.4,1199.0,55.0


Query 2: 'men slim fit chinos' | word2vec


,pid,title,score,rating,price,discount
rank,,,,,,
1,TROE3N2FAAGV3HSG,Rocky Slim Fit Men Beige Cotton Blend Trousers,0.9121,3.6,450.0,49.0
2,TROE3N2FYUQWF8DX,Rocky Slim Fit Men Beige Cotton Blend Trousers,0.9121,3.6,496.0,44.0
3,TRODYPCYVYUZH3MZ,Slim Fit Men Yellow Cotton Blend Trousers,0.9102,3.7,699.0,70.0
4,TRODZZMJYWQTEGRY,Rocky Slim Fit Men Black Cotton Blend Trousers,0.9098,3.6,499.0,44.0
5,TROE3N2GXGYWPZ3H,Rocky Slim Fit Men Silver Cotton Blend Trousers,0.9064,3.6,456.0,49.0
6,TRODYPCYQNWMZ7G2,Slim Fit Men Beige Cotton Blend Trousers,0.9060,3.6,352.0,64.0
7,TRODYPCYT5FT6BJT,Slim Fit Men Beige Cotton Blend Trousers,0.9060,3.6,299.0,70.0
8,TRODYPCY2XCDQNTT,Slim Fit Men Brown Cotton Blend Trousers,0.9041,3.6,499.0,50.0
9,TROE3N2GDGHXPYYJ,"Rocky Slim Fit Men Silver, Beige Cotton Blend Trousers",0.9041,3.3,999.0,44.0


### 2.3 Query 3: hooded full sleeve sweatshirt

In [7]:
show_query(3)

Query 3: 'hooded full sleeve sweatshirt' | tfidf


,pid,title,score,rating,price,discount
rank,,,,,,
1,SWSFZ2H5XK7TJZRS,Full Sleeve Solid Men Sweatshirt,0.7230,NaN,730.0,66.0
2,SWSFZ2H3CXFAWYPH,Full Sleeve Solid Women Sweatshirt,0.7216,2.5,752.0,65.0
3,SWSFXV6GGZWJEDKJ,Full Sleeve Solid Men Sweatshirt,0.7214,3.5,742.0,67.0
4,SWSFY37MCKGY6GHQ,Full Sleeve Solid Women Sweatshirt,0.7185,3.6,725.0,67.0
5,SWSFY37VGGQGSZ86,Full Sleeve Solid Women Sweatshirt,0.7185,2.8,703.0,68.0
6,SWSFY382ZD3YXMVP,Full Sleeve Solid Men Sweatshirt,0.7184,3.8,687.0,68.0
7,SWSFZ2H2T8SEC45H,Full Sleeve Solid Men Sweatshirt,0.7184,2.5,747.0,66.0
8,SWSFZ2H5RNZFAZWW,Full Sleeve Solid Men Sweatshirt,0.7184,2.5,719.0,68.0
9,SWSFZ2H5ZGRP6DY2,Full Sleeve Solid Men Sweatshirt,0.7184,4.7,742.0,67.0


Query 3: 'hooded full sleeve sweatshirt' | bm25


,pid,title,score,rating,price,discount
rank,,,,,,
1,SWSFXZWGFKHNG88S,Full Sleeve Solid Men Sweatshirt,13.1303,3.2,649.0,63.0
2,SWSFXZWGMGVDUCXA,Full Sleeve Printed Women Sweatshirt,13.1303,3.2,649.0,63.0
3,SWSFXZWWYXPZXPTY,Full Sleeve Printed Men Sweatshirt,13.1303,3.2,649.0,63.0
4,SWSFYV4N7HJ8BHWM,Full Sleeve Solid Men Sweatshirt,13.1303,3.2,649.0,63.0
5,SWSFYV4NUHK8YPR9,Full Sleeve Solid Men Sweatshirt,13.1303,3.2,679.0,66.0
6,SWSFYV85ZTT9H7ZY,Full Sleeve Solid Men Sweatshirt,13.1303,3.2,679.0,66.0
7,SWSFXMFP2RUM86QD,Full Sleeve Printed Women Sweatshirt,13.0371,3.7,749.0,42.0
8,SWSFXMFPRTY4GKGF,Full Sleeve Printed Men Sweatshirt,13.0371,3.7,949.0,50.0
9,SWSFXMFPVYZ2YGGQ,Full Sleeve Embroidered Men Sweatshirt,13.0371,3.7,849.0,59.0


Query 3: 'hooded full sleeve sweatshirt' | custom


,pid,title,score,rating,price,discount
rank,,,,,,
1,SWSFPHFFKEGXGVHW,Full Sleeve Solid Men Sweatshirt,1.2702,4.0,799.0,65.0
2,SWSFXMHDY9ZHDAW9,Full Sleeve Embroidered Women Sweatshirt,1.2517,3.7,699.0,66.0
3,SWSFXMFPVYZ2YGGQ,Full Sleeve Embroidered Men Sweatshirt,1.2448,3.7,849.0,59.0
4,SWSFVNAYAGBGKAPZ,Full Sleeve Ombre Men Sweatshirt,1.2426,4.6,949.0,50.0
5,SWSFXMFPRTY4GKGF,Full Sleeve Printed Men Sweatshirt,1.2359,3.7,949.0,50.0
6,SWSFYV4NUHK8YPR9,Full Sleeve Solid Men Sweatshirt,1.2310,3.2,679.0,66.0
7,SWSFYV85ZTT9H7ZY,Full Sleeve Solid Men Sweatshirt,1.2310,3.2,679.0,66.0
8,SWSFXMFPYYTZUZ3Z,Full Sleeve Printed Men Sweatshirt,1.2294,3.7,799.0,60.0
9,SWSFXMFP2RUM86QD,Full Sleeve Printed Women Sweatshirt,1.2281,3.7,749.0,42.0


Query 3: 'hooded full sleeve sweatshirt' | word2vec


,pid,title,score,rating,price,discount
rank,,,,,,
1,SWSFVZPSTS5UQ8YN,Full Sleeve Solid Women Sweatshirt,0.9118,4.2,890.0,44.0
2,SWSFYE8BSGHCMCCG,Full Sleeve Printed Men Sweatshirt,0.9073,4.1,699.0,65.0
3,SWSFYE8DVWRPTQWH,Full Sleeve Printed Men Sweatshirt,0.9071,4.1,699.0,65.0
4,SWSFXMHDY9ZHDAW9,Full Sleeve Embroidered Women Sweatshirt,0.9039,3.7,699.0,66.0
5,SWSFXMKSWR9XWXHS,Full Sleeve Color Block Women Sweatshirt,0.9023,2.8,849.0,50.0
6,SWSFPHFFKEGXGVHW,Full Sleeve Solid Men Sweatshirt,0.9019,4.0,799.0,65.0
7,SWSFXZWWYXPZXPTY,Full Sleeve Printed Men Sweatshirt,0.9009,3.2,649.0,63.0
8,SWSFXMFPVYZ2YGGQ,Full Sleeve Embroidered Men Sweatshirt,0.8999,3.7,849.0,59.0
9,SWSFYE8HGBSPAYES,Full Sleeve Printed Women Sweatshirt,0.8997,4.0,699.0,65.0


### 2.4 Query 4: blue denim shorts

In [8]:
show_query(4)

Query 4: 'blue denim shorts' | tfidf


,pid,title,score,rating,price,discount
rank,,,,,,
1,SRTFEN3WSH8JHFHN,Solid Women Blue Denim Shorts,0.9625,4.5,5990.0,0.0
2,SRTFZGPJFAA6D9NA,Solid Women Denim Blue Denim Shorts,0.9140,4.0,749.0,50.0
3,SRTFVEQPDREMKKBW,Solid Women Denim Blue Denim Shorts,0.8798,2.9,599.0,50.0
4,SRTFZAHDNTTGS3HF,Self Design Men Denim Dark Blue Denim Shorts,0.6792,3.5,1519.0,58.0
5,SRTFVCCCSEWS46YH,Washed Women Denim Blue Denim Shorts,0.5436,3.8,799.0,60.0
6,SRTFVC47TFYZEFDP,Washed Women Denim Dark Blue Denim Shorts,0.5346,3.8,799.0,60.0
7,SRTF8S4GSG7HSVDZ,Solid Women Blue Denim Shorts,0.5073,3.8,5990.0,0.0
8,TSHFNHJVGGHMPEWX,Printed Men Round Neck Dark Blue T-Shirt,0.5025,3.2,900.0,57.0
9,SHTFJ2GMHK5ZTZDB,Women Slim Fit Self Design Slim Collar Casual Shirt,0.4855,2.8,891.0,55.0


Query 4: 'blue denim shorts' | bm25


,pid,title,score,rating,price,discount
rank,,,,,,
1,SRTFVEQPDREMKKBW,Solid Women Denim Blue Denim Shorts,10.9812,2.9,599.0,50.0
2,SRTFZGPJFAA6D9NA,Solid Women Denim Blue Denim Shorts,10.8184,4.0,749.0,50.0
3,SRTFEN3WSH8JHFHN,Solid Women Blue Denim Shorts,10.1419,4.5,5990.0,0.0
4,SRTFZAHDNTTGS3HF,Self Design Men Denim Dark Blue Denim Shorts,10.0914,3.5,1519.0,58.0
5,SRTFVCCCSEWS46YH,Washed Women Denim Blue Denim Shorts,9.7463,3.8,799.0,60.0
6,SRTFVC47TFYZEFDP,Washed Women Denim Dark Blue Denim Shorts,9.6502,3.8,799.0,60.0
7,SRTF8S4GSG7HSVDZ,Solid Women Blue Denim Shorts,9.5934,3.8,5990.0,0.0
8,SRTFKJZN76QFHFHK,Distressed Women Denim Dark Blue Denim Shorts,9.3151,3.8,1237.0,66.0
9,SRTFKJPPFQKRJ4XS,Distressed Women Denim Dark Blue Denim Shorts,9.2024,4.0,1261.0,65.0


Query 4: 'blue denim shorts' | custom


,pid,title,score,rating,price,discount
rank,,,,,,
1,SRTFZGPJFAA6D9NA,Solid Women Denim Blue Denim Shorts,1.2490,4.0,749.0,50.0
2,SRTFVEQPDREMKKBW,Solid Women Denim Blue Denim Shorts,1.1925,2.9,599.0,50.0
3,SRTFEN3WSH8JHFHN,Solid Women Blue Denim Shorts,1.1297,4.5,5990.0,0.0
4,SRTFZAHDNTTGS3HF,Self Design Men Denim Dark Blue Denim Shorts,1.1066,3.5,1519.0,58.0
5,SRTFVCCCSEWS46YH,Washed Women Denim Blue Denim Shorts,1.0735,3.8,799.0,60.0
6,SRTFVC47TFYZEFDP,Washed Women Denim Dark Blue Denim Shorts,1.0582,3.8,799.0,60.0
7,SRTFKJPPFQKRJ4XS,Distressed Women Denim Dark Blue Denim Shorts,1.0024,4.0,1261.0,65.0
8,SRTF8S4GSG7HSVDZ,Solid Women Blue Denim Shorts,0.9996,3.8,5990.0,0.0
9,KTAFZHX59PEWDGCR,Men Solid Denim Straight Kurta (Blue),0.8806,3.3,809.0,59.0


Query 4: 'blue denim shorts' | word2vec


,pid,title,score,rating,price,discount
rank,,,,,,
1,SRTFVEQPDREMKKBW,Solid Women Denim Blue Denim Shorts,0.9701,2.9,599.0,50.0
2,SRTFZGPJFAA6D9NA,Solid Women Denim Blue Denim Shorts,0.9618,4.0,749.0,50.0
3,SRTFEN3WSH8JHFHN,Solid Women Blue Denim Shorts,0.9014,4.5,5990.0,0.0
4,SRTFZAHDNTTGS3HF,Self Design Men Denim Dark Blue Denim Shorts,0.8926,3.5,1519.0,58.0
5,SRTF8S4GSG7HSVDZ,Solid Women Blue Denim Shorts,0.8559,3.8,5990.0,0.0
6,TFHE9EW7BRTAVXUX,Solid Men Denim Three Fourths,0.8528,3.7,799.0,0.0
7,SRTFVC47TFYZEFDP,Washed Women Denim Dark Blue Denim Shorts,0.8506,3.8,799.0,60.0
8,SRTFVCCCSEWS46YH,Washed Women Denim Blue Denim Shorts,0.8501,3.8,799.0,60.0
9,TFHE9EW7UEEEUZST,Solid Men Denim Three Fourths,0.8489,3.7,799.0,0.0


### 2.5 Query 5: women black leggings

In [9]:
show_query(5)

Query 5: 'women black leggings' | tfidf


,pid,title,score,rating,price,discount
rank,,,,,,
1,TKPFGZYS9CWTRMZJ,Solid Women Black Track Pants,0.2817,3.3,1448.0,37.0
2,JEAFANJNYQRETCSN,Stretchable Slim Women Black Jeans (Pack of 2),0.2483,3.7,1275.0,57.0
3,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,0.2414,3.0,1484.0,55.0
4,TKPFK3W6VQFCYNTQ,Solid Women Black Track Pants,0.2302,4.2,2017.0,32.0
5,SRTFGZY9UCFAAEHM,Solid Women Black Sports Shorts,0.2106,4.3,1985.0,33.0
6,TKPFGZYSMFQGDESU,Solid Women Black Track Pants,0.1959,3.4,2999.0,50.0
7,TKPF4GGXRGJYNJVE,"Printed, Solid Women Grey Track Pants",0.1703,2.3,1199.0,67.0
8,TKPF4GGX6TAYKPCQ,"Printed, Solid Women Grey Track Pants",0.1703,2.3,1199.0,67.0
9,TKPFUY8XUNRHPNDZ,Printed Women Black Track Pants,0.1609,4.3,2144.0,35.0


Query 5: 'women black leggings' | bm25


,pid,title,score,rating,price,discount
rank,,,,,,
1,TKPFGZYS9CWTRMZJ,Solid Women Black Track Pants,8.7400,3.3,1448.0,37.0
2,TKPFGZYSMFQGDESU,Solid Women Black Track Pants,8.5783,3.4,2999.0,50.0
3,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,8.4998,3.0,1484.0,55.0
4,TKPFK3W6VQFCYNTQ,Solid Women Black Track Pants,8.4227,4.2,2017.0,32.0
5,SRTFGZY9UCFAAEHM,Solid Women Black Sports Shorts,7.9888,4.3,1985.0,33.0
6,JEAFANJNYQRETCSN,Stretchable Slim Women Black Jeans (Pack of 2),7.3124,3.7,1275.0,57.0
7,TKPFN5S8QKWKFHCM,Solid Women Black Track Pants,6.9756,4.9,4499.0,0.0
8,TKPF4GGX6TAYKPCQ,"Printed, Solid Women Grey Track Pants",6.8832,2.3,1199.0,67.0
9,TKPF4GGXRGJYNJVE,"Printed, Solid Women Grey Track Pants",6.8832,2.3,1199.0,67.0


Query 5: 'women black leggings' | custom


,pid,title,score,rating,price,discount
rank,,,,,,
1,TKPFGZYSMFQGDESU,Solid Women Black Track Pants,1.1707,3.4,2999.0,50.0
2,TKPFK3W6VQFCYNTQ,Solid Women Black Track Pants,1.1516,4.2,2017.0,32.0
3,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,1.1187,3.0,1484.0,55.0
4,SRTFGZY9UCFAAEHM,Solid Women Black Sports Shorts,0.9936,4.3,1985.0,33.0
5,JEAFANJNYQRETCSN,Stretchable Slim Women Black Jeans (Pack of 2),0.7232,3.7,1275.0,57.0
6,TKPFN5S8QKWKFHCM,Solid Women Black Track Pants,0.6124,4.9,4499.0,0.0
7,TKPF4GGX6TAYKPCQ,"Printed, Solid Women Grey Track Pants",0.5196,2.3,1199.0,67.0
8,TKPF4GGXRGJYNJVE,"Printed, Solid Women Grey Track Pants",0.5196,2.3,1199.0,67.0
9,TKPFUY8XUNRHPNDZ,Printed Women Black Track Pants,0.5051,4.3,2144.0,35.0


Query 5: 'women black leggings' | word2vec


,pid,title,score,rating,price,discount
rank,,,,,,
1,TKPFK3W6NVCGUTHR,Solid Women Black Track Pants,0.6078,3.0,1484.0,55.0
2,TROEWT2N4GNNPBHM,Slim Fit Women Multicolor Polycotton Trousers,0.5844,3.9,570.0,40.0
3,TKPFGZYSMFQGDESU,Solid Women Black Track Pants,0.5815,3.4,2999.0,50.0
4,TROEWT27ZGBUPCZM,Slim Fit Women Multicolor Polycotton Trousers,0.5634,3.3,590.0,40.0
5,SRTFGZY9UCFAAEHM,Solid Women Black Sports Shorts,0.5449,4.3,1985.0,33.0
6,TKPF4GGXRGJYNJVE,"Printed, Solid Women Grey Track Pants",0.5257,2.3,1199.0,67.0
7,TKPF4GGX6TAYKPCQ,"Printed, Solid Women Grey Track Pants",0.5185,2.3,1199.0,67.0
8,TKPFN5S8QKWKFHCM,Solid Women Black Track Pants,0.4856,4.9,4499.0,0.0
9,TKPFK3W6VQFCYNTQ,Solid Women Black Track Pants,0.4758,4.2,2017.0,32.0


## 3. Comparison between methods

Overlap: number of documents shared by the top 20 of two methods. Kendall tau between TF-IDF and BM25 over the top 20 (documents in the union of both top 20, ranked by their position in each full ranking) and over all the AND results.

In [10]:
def kendall_tfidf_bm25(ranking, top_k=None):
    pos_tfidf = {pid: i for i, (pid, _) in enumerate(ranking["tfidf"])}
    pos_bm25 = {pid: i for i, (pid, _) in enumerate(ranking["bm25"])}
    if top_k:
        pids = sorted({p for p, _ in ranking["tfidf"][:top_k]} | {p for p, _ in ranking["bm25"][:top_k]})
    else:
        pids = sorted(pos_tfidf)
    if len(pids) < 2:
        return float("nan")
    return kendalltau([pos_tfidf[p] for p in pids], [pos_bm25[p] for p in pids]).statistic


comparison = []
for qid, query in enumerate(TEST_QUERIES, start=1):
    ranking = rankings[query]
    tops = {m: {pid for pid, _ in ranking[m][:TOP_K]} for m in METHODS}
    row = {"query_id": qid, "query": query, "top size": len(tops["tfidf"])}
    for a, b in combinations(METHODS, 2):
        row[f"{a} & {b}"] = len(tops[a] & tops[b])
    row["tau tfidf-bm25 (top 20)"] = kendall_tfidf_bm25(ranking, TOP_K)
    row["tau tfidf-bm25 (all)"] = kendall_tfidf_bm25(ranking)
    comparison.append(row)
pd.DataFrame(comparison).set_index("query_id").round(3)

,query,top size,tfidf & bm25,tfidf & custom,tfidf & word2vec,bm25 & custom,bm25 & word2vec,custom & word2vec,tau tfidf-bm25 (top 20),tau tfidf-bm25 (all)
query_id,,,,,,,,,,
1,women cotton kurta,20,12,6,8,10,7,3,-0.201,0.577
2,men slim fit chinos,20,14,11,4,15,8,8,0.071,0.810
3,hooded full sleeve sweatshirt,20,3,3,1,18,12,12,-0.468,0.716
4,blue denim shorts,20,18,17,18,18,18,18,0.506,0.865
5,women black leggings,13,13,13,13,13,13,13,0.692,0.692


## 4. Weights of the custom score

custom = BM25 min-max normalised over the AND results x (1 + w_rating * (rating - 1) / 4 + w_discount * discount / 100).
Products without rating get the corpus mean rating and out of stock products always go after all the in-stock ones.

We compare four weight sets (w_rating, w_discount). B is the default in src/ranking.py (CUSTOM_WEIGHTS).

- A: 0.1, 0.05
- B: 0.3, 0.1
- C: 0.5, 0.2
- D: 1.0, 0.3

In [11]:
WEIGHT_SETS = {"A": (0.1, 0.05), "B": (0.3, 0.1), "C": (0.5, 0.2), "D": (1.0, 0.3)}
stock = corpus.set_index("pid")["out_of_stock"]


def rank_weight_sets(query):
    """BM25 and custom ranking of a query with every weight set: name -> [pid]."""
    pids = conjunctive_search(query, index)
    runs = {"bm25": [p for p, _ in rank_bm25(query, pids, index, idf_bm25, doc_lens, avg_len)]}
    for name, (w_rating, w_discount) in WEIGHT_SETS.items():
        weights = {"rating": w_rating, "discount": w_discount}
        runs[name] = [p for p, _ in rank_custom(query, pids, index, idf_bm25, doc_lens, avg_len,
                                                product_info, weights)]
    return runs


weight_runs = {q: rank_weight_sets(q) for q in TEST_QUERIES}

# How many of the BM25 top 10 are still in the top 10 of every weight set
overlap = []
for qid, query in enumerate(TEST_QUERIES, start=1):
    top_bm25 = set(weight_runs[query]["bm25"][:10])
    row = {"query_id": qid, "query": query}
    row.update({name: len(top_bm25 & set(weight_runs[query][name][:10])) for name in WEIGHT_SETS})
    overlap.append(row)
pd.DataFrame(overlap).set_index("query_id")

,query,A,B,C,D
query_id,,,,,
1,women cotton kurta,6,4,4,2
2,men slim fit chinos,8,8,7,7
3,hooded full sleeve sweatshirt,9,7,5,5
4,blue denim shorts,9,9,9,9
5,women black leggings,9,9,9,9


In [12]:
def show_weight_sets(qid, top_k=10):
    """Top k of BM25 and of every weight set side by side: title, rating, discount and stock."""
    query = TEST_QUERIES[qid - 1]
    print(f"Query {qid}: '{query}'")
    columns = {}
    for name, ranking in weight_runs[query].items():
        top = ranking[:top_k]
        columns[(name, "title")] = products.loc[top, "title"].str[:35].values
        columns[(name, "rating")] = products.loc[top, "rating"].values
        columns[(name, "discount")] = products.loc[top, "discount"].values
        columns[(name, "stock")] = ["out" if stock[p] else "in" for p in top]
    display(pd.DataFrame(columns, index=range(1, len(top) + 1)))

### 4.1 Query 1: women cotton kurta

In [13]:
show_weight_sets(1)

Query 1: 'women cotton kurta'


bm25                        \
                                  title rating discount stock   
1   Women Striped Cotton Blend Straight    4.3     70.0    in   
2   Women Solid Cotton Blend Straight K    4.0     79.0    in   
3   Women Printed Pure Cotton Straight     4.1     64.0    in   
4   Women Solid Cotton Blend Pathani Ku    4.0     74.0    in   
5   Women Solid Cotton Blend Pathani Ku    3.6     75.0    in   
6   Women Solid Cotton Blend Straight K    3.8     72.0    in   
7   Women Kurta and Pyjama Set Cotton B    3.0     35.0    in   
8   Women Striped Cotton Blend Straight    3.0     39.0    in   
9   Women Printed Cotton Blend Straight    4.1     33.0    in   
10  Women Striped Cotton Blend Straight    3.3     72.0    in   

                                      A                        \
                                  title rating discount stock   
1   Women Striped Cotton Blend Straight    4.3     70.0    in   
2   Women Solid Cotton Blend Straight K    4.0     79.0    in   
3   Women Printed Pure Cotton Straight     4.1     64.0    in   
4   Women Solid Cotton Blend Pathani Ku    4.0     74.0    in   
5   Women Solid Cotton Blend Pathani Ku    3.6     75.0    in   
6   Women Solid Cotton Blend Straight K    3.8     72.0    in   
7   Women Self Design Cotton Linen Blen    4.1     73.0    in   
8   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
9   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
10  Women Solid Pure Cotton Straight Ku    4.7     65.0    in   

                                      B                        \
                                  title rating discount stock   
1   Women Striped Cotton Blend Straight    4.3     70.0    in   
2   Women Solid Cotton Blend Straight K    4.0     79.0    in   
3   Women Printed Pure Cotton Straight     4.1     64.0    in   
4   Women Solid Cotton Blend Straight K    5.0     64.0    in   
5   Women Solid Cotton Blend Straight K    5.0     64.0    in   
6   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
7   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
8   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
9   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
10  Women Solid Cotton Blend Pathani Ku    4.0     74.0    in   

                                      C                        \
                                  title rating discount stock   
1   Women Striped Cotton Blend Straight    4.3     70.0    in   
2   Women Solid Cotton Blend Straight K    4.0     79.0    in   
3   Women Solid Cotton Blend Straight K    5.0     64.0    in   
4   Women Solid Cotton Blend Straight K    5.0     64.0    in   
5   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
6   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
7   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
8   Women Solid Pure Cotton Straight Ku    4.7     65.0    in   
9   Women Printed Pure Cotton Straight     4.1     64.0    in   
10  Women Solid Cotton Blend Pathani Ku    4.0     74.0    in   

                                      D                        
                                  title rating discount stock  
1   Women Striped Cotton Blend Straight    4.3     70.0    in  
2   Women Solid Cotton Blend Straight K    5.0     64.0    in  
3   Women Solid Cotton Blend Straight K    5.0     64.0    in  
4   Women Solid Pure Cotton Straight Ku    4.7     65.0    in  
5   Women Solid Pure Cotton Straight Ku    4.7     65.0    in  
6   Women Solid Pure Cotton Straight Ku    4.7     65.0    in  
7   Women Solid Pure Cotton Straight Ku    4.7     65.0    in  
8   Women Solid Cotton Blend Straight K    4.0     79.0    in  
9   Women Solid Cotton Blend Straight K    5.0     64.0    in  
10  Women Kurta and Pyjama Set Cotton B    4.8     36.0    in

### 4.2 Query 3: hooded full sleeve sweatshirt

In [14]:
show_weight_sets(3)

Query 3: 'hooded full sleeve sweatshirt'

bm25                        \
                                  title rating discount stock   
1      Full Sleeve Solid Men Sweatshirt    3.2     63.0    in   
2   Full Sleeve Printed Women Sweatshir    3.2     63.0    in   
3    Full Sleeve Printed Men Sweatshirt    3.2     63.0    in   
4      Full Sleeve Solid Men Sweatshirt    3.2     63.0    in   
5      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
6      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
7   Full Sleeve Printed Women Sweatshir    3.7     42.0    in   
8    Full Sleeve Printed Men Sweatshirt    3.7     50.0    in   
9   Full Sleeve Embroidered Men Sweatsh    3.7     59.0    in   
10  Full Sleeve Embroidered Women Sweat    3.7     66.0    in   

                                      A                        \
                                  title rating discount stock   
1      Full Sleeve Solid Men Sweatshirt    4.0     65.0    in   
2      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
3      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
4      Full Sleeve Solid Men Sweatshirt    3.2     63.0    in   
5   Full Sleeve Printed Women Sweatshir    3.2     63.0    in   
6    Full Sleeve Printed Men Sweatshirt    3.2     63.0    in   
7      Full Sleeve Solid Men Sweatshirt    3.2     63.0    in   
8   Full Sleeve Embroidered Women Sweat    3.7     66.0    in   
9   Full Sleeve Embroidered Men Sweatsh    3.7     59.0    in   
10   Full Sleeve Printed Men Sweatshirt    3.7     50.0    in   

                                      B                        \
                                  title rating discount stock   
1      Full Sleeve Solid Men Sweatshirt    4.0     65.0    in   
2   Full Sleeve Embroidered Women Sweat    3.7     66.0    in   
3   Full Sleeve Embroidered Men Sweatsh    3.7     59.0    in   
4      Full Sleeve Ombre Men Sweatshirt    4.6     50.0    in   
5    Full Sleeve Printed Men Sweatshirt    3.7     50.0    in   
6      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
7      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
8    Full Sleeve Printed Men Sweatshirt    3.7     60.0    in   
9   Full Sleeve Printed Women Sweatshir    3.7     42.0    in   
10     Full Sleeve Solid Men Sweatshirt    3.2     63.0    in   

                                      C                        \
                                  title rating discount stock   
1      Full Sleeve Solid Men Sweatshirt    4.0     65.0    in   
2      Full Sleeve Ombre Men Sweatshirt    4.6     50.0    in   
3   Full Sleeve Embroidered Women Sweat    3.7     66.0    in   
4   Full Sleeve Embroidered Men Sweatsh    3.7     59.0    in   
5    Full Sleeve Printed Men Sweatshirt    3.7     60.0    in   
6    Full Sleeve Printed Men Sweatshirt    3.7     50.0    in   
7      Full Sleeve Solid Men Sweatshirt    3.8     72.0    in   
8    Full Sleeve Solid Women Sweatshirt    3.7     54.0    in   
9      Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   
10     Full Sleeve Solid Men Sweatshirt    3.2     66.0    in   

                                      D                        
                                  title rating discount stock  
1      Full Sleeve Ombre Men Sweatshirt    4.6     50.0    in  
2      Full Sleeve Solid Men Sweatshirt    4.0     65.0    in  
3   Full Sleeve Embroidered Women Sweat    3.7     66.0    in  
4   Full Sleeve Embroidered Men Sweatsh    3.7     59.0    in  
5      Full Sleeve Solid Men Sweatshirt    3.8     72.0    in  
6    Full Sleeve Printed Men Sweatshirt    3.7     60.0    in  
7    Full Sleeve Printed Men Sweatshirt    3.7     50.0    in  
8    Full Sleeve Solid Women Sweatshirt    3.7     54.0    in  
9   Full Sleeve Printed Women Sweatshir    3.7     42.0    in  
10     Full Sleeve Solid Men Sweatshirt    3.2     66.0    in